In [1]:
import os
os.environ['OMP_NUM_THREADS'] = '1'

import sys
import re
import json
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import sklearn
from scipy import signal
import ddddocr
import torch
from torchvision.models import resnet18, ResNet18_Weights

In [ ]:
# 全局配置
class NanokaDetector():
    def __init__(self, image):
        super(NanokaDetector, self).__init__()

        self.image = image
        self.gray = cv2.cvtColor(self.image, cv2.COLOR_BGR2GRAY)
        self.ocr = ddddocr.DdddOcr(show_ad=False)
        self.size_info = []

    def update_image(self, image):
        '''
        更新使用的图片, 我们可以直接调取这个图片进行分析
        '''
        self.image = image
        self.gray = cv2.cvtColor(self.image, cv2.COLOR_BGR2GRAY)
                         
    # 亮点综合检测器
    def threhold_detector(self, plot=False, show=False):
        '''
        利用检测点的圆形高亮特性确定目标点
        '''
        # 一顿操作最后只剩下需要的信息
        _, binary_image = cv2.threshold(self.gray, 155, 255, cv2.THRESH_BINARY)
        blured_image = cv2.blur(binary_image, (9,9))
        median_image = cv2.medianBlur(blured_image, 7)
        _, rebinary_image = cv2.threshold(median_image, 155, 255, cv2.THRESH_BINARY)
        
        # 掩膜
        height, width = self.gray.shape
        border_ratio = 0.1
        mask = np.zeros_like(self.gray, dtype=np.uint8)
        border_x = int(width * border_ratio)
        border_y = int(height * border_ratio)
        inner_width = width - 2 * border_x
        inner_height = height - 2 * border_y
        cv2.rectangle(mask, (border_x, border_y), (border_x + inner_width, border_y + inner_height), 255, -1)
        masked_rebinary_image = cv2.bitwise_and(mask, rebinary_image, mask=mask)
        
        # 提取大范围亮色特征 (轮廓分析)
        targets, anti_targets = [], []
        origin_image = None
        if plot:
            origin_image = self.image.copy()
            # 绘制掩膜
            cv2.rectangle(origin_image, (border_x, border_y), (border_x + inner_width, border_y + inner_height), 255, 2)
    
        # 抓住所有需要的点特征
        contours, _ = cv2.findContours(masked_rebinary_image, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for contour in contours:
            x, y, w, h = cv2.boundingRect(contour)
            center_x, center_y = x + w // 2, y + h // 2
            targets.append((center_x, center_y, w, h))
            
            if plot:
                cv2.rectangle(origin_image, (x, y), (x+w, y+h), (0, 255, 0), 2)  # 绿色矩形
                cv2.circle(origin_image, (center_x, center_y), 5, (255, 0, 0), -1)  # 红色点

        # 消除不需要的点特征
        masked_median_image = cv2.medianBlur(cv2.medianBlur(masked_rebinary_image, 11), 11)
        acontours, _ = cv2.findContours(masked_median_image, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for contour in acontours:
            x, y, w, h = cv2.boundingRect(contour)
            center_x, center_y = x + w // 2, y + h // 2
            anti_targets.append((center_x, center_y, w, h))
            
            if plot:
                cv2.rectangle(origin_image, (x-5, y-5), (x+w+5, y+h+5), (255, 0, 0), 2)  # 红色矩形
    
        if show:
            fig, axes = plt.subplots(1, 1, figsize=(10, 6))
            axes.imshow(origin_image)
            axes.set_title("Analyzed Image")
            plt.show()
        
        return targets, anti_targets, origin_image

    def ocr_detector(border_ratio=0.1):
        '''
        ddddocr 图像文字识别
        '''
        height, width = self.gray.shape
        mask = np.zeros_like(self.gray, dtype=np.uint8)
        border_x = int(width * border_ratio)
        border_y = int(height * border_ratio)
        
        text_area = self.gray[height-border_y:, :border_x]    
        _, encoded_img = cv2.imencode('.png', text_area)
        img_bytes = encoded_img.tobytes()
    
        # 只能检测编码字节流
        result = self.ocr.classification(img_bytes)
        return result

    def re_keyword_detector(texts):
        """
        ASCII字符清洗检测子串, 子串是提前规定好的
        """
        patterns = [r'添[\x00-\x7F]{0,3}加[\x00-\x7F]{0,3}好[\x00-\x7F]{0,3}友', r'好[\x00-\x7F]{0,3}友', r'挚[\x00-\x7F]{0,3}友']
        dataframes = {
            '识别文本': [],
            '星盘页': [],
            '添加好友': [],
            '好友': [],
            '挚友': [],
        }
    
        for text in texts:
            matchs = list([bool(re.search(pattern, text)) for pattern in patterns])
            dataframes['星盘页'].append(True in matchs)
            dataframes['添加好友'].append(matchs[0] == True)
            dataframes['好友'].append(matchs[1] == True)
            dataframes['挚友'].append(matchs[2] == True)
            dataframes['识别文本'].append(text)
            
        return pd.DataFrame(dataframes)

    # 圆形检测器
    def hough_detector(self, plot=False, show=False):
        '''
        使用霍夫变换检测圆形目标, 为后续提供支持
        '''
        min_radius=1
        max_radius=35
    
        # 使用霍夫圆变换检测圆形
        circles = cv2.HoughCircles(
            masked_edges,               # 输入图像（边缘图）
            cv2.HOUGH_GRADIENT,         # 检测方法（梯度法）
            dp=1,                       # 累加器图像的分辨率与原图之比
            minDist=50,                 # 检测到的圆的圆心之间的最小距离
            param1=50,                  # Canny边缘检测器的高阈值
            param2=25,                  # 累加器阈值（越小检测到的圆越多）
            minRadius=min_radius,       # 最小圆半径
            maxRadius=max_radius        # 最大圆半径
        )
    
        # 复制原图用于绘制结果
        origin_image = None
        if plot:
            origin_image = self.image.copy()
            cv2.rectangle(origin_image, (border_x, border_y), (border_x + inner_width, border_y + inner_height), 255, 2)

        if circles is not None:
            circles = np.round(circles[0, :]).astype("int")
            if plot:
                for (x, y, r) in circles:
                    cv2.circle(origin_image, (x, y), r, (0, 255, 0), 4)
        else:
            print("No detection.")
            
        if show:
            fig, axes = plt.subplots(1, 1, figsize=(10, 6))
            axes.imshow(origin_image, cmap="gray")
            axes.set_title("Masked Hough transform Image")
            plt.show()
            
        return circles, origin_image


    def image_spiltor(self):
        '''
        用于将数据切分成我们想要的形式之后进行分类任务就可以了
        '''
        targets, anti_targets, plot_image = self.threhold_detector(plot=False, show=False)

        diff = np.expand_dims(targets[:,:2], axis=1) - np.expand_dims(anti_targets[:,:2], axis=0)
        distance = np.sqrt(np.sum(diff*diff, axis=2))
        point = np.where(distance < 15)[0]
        post = copy(targets[point])
        pre = [i for i in targets.tolist() if i not in post]

        # 不同的类别进行不同的切割方式, pre 是需要进一步处理的内容, post是确定的很大的不需要处理的内容
        pre_scaler, post_scaler = 5, 2
        pre_sub_gray = [gray[int(y-max(w,h)*pre_scaler):int(y+max(w, h)*pre_scaler), int(x-max(w, h)*pre_scaler):int(x+max(w, h)*pre_scaler)] for x, y, w, h in pre]
        post_sub_gray = [gray[int(y-max(w,h)*post_scaler):int(y+max(w, h)*post_scaler), int(x-max(w, h)*post_scaler):int(x+max(w, h)*post_scaler)] for x, y, w, h in post]
        return pre_sub_gray, post_sub_gray
        
    def multi_detector(self, border_ratio=0.1, plot=False, show=False):
        '''
        多重鉴别器, 用于检测是否在星盘页并且判断类型
        '''
        text = ocr_detector(self.gray)
        pf = re_keyword_detector([text])
    
        if not bool(pf['星盘页'].values[0]):
            print("不在星盘页, 不进行后续检测")
            if show:
                fig, axes = plt.subplots(1, 1, figsize=(15, 5))
                axes.imshow(self.image)
                axes.set_title("Useless Image")
                axes.axis('off')
                plt.show()
            return {"code":-1, "info":"不在星盘页"}, None
    
        if bool(pf['添加好友'].values[0]):
            print("当前页面为添加好友页")
            if show:
                fig, axes = plt.subplots(1, 1, figsize=(15, 5))
                axes.imshow(self.image)
                axes.set_title("Make Friends Image")
                axes.axis('off')
                plt.show()
            return {"code":1, "info":"本页面为添加好友页"}, None
    
        if bool(pf['好友'].values[0]) or bool(pf['挚友'].values[0]):
            print("检测为有效的星盘页")
            tfa_targets, img =  tfa_detector(self.gray, plot=True, show=False)
            if show:
                r_list = np.array(sorted([target[1][2] for target in tfa_targets]))
                fig, axes = plt.subplots(1, 2, figsize=(15, 5))
                plt.grid(True, linestyle="--", alpha=1)
                sns.lineplot(x="Count", y="Radius", label="Radius", data={"Count":np.arange(1, r_list.shape[0]+1, 1), "Radius": r_list}, ax=axes[0])
                axes[0].set_title("Radius Plot Image")
                axes[1].imshow(img)
                axes[1].set_title("TFA Detector Result")
                axes[1].axis('off')
                plt.grid(False, linestyle="--", alpha=1)
                plt.tight_layout()
                plt.show()
            return {"code":0, "info":"识别成功", "tfa_targets":tfa_targets}, img
        return {"code":-2, "info":"无法预知的错误!"}, None

In [ ]:
for filename in os.listdir(os.path.join("source", "data")):
    if filename.endswith(".png") or filename.endswith(".jpg"):
        print(f"\n\nProcessing {filename}...")
        
        image = cv2.imread(os.path.join("source", "data", filename))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
        
        json_data, img = multi_detector(gray, plot=True, show=True)
    else:
        print(f"Skipping {filename}, not an image file.")